### FastText
- Word2Vec에서 OOV(Out Of Voca) 문제를 해결하기 위한 모델 
- Word2Vec은 '강아지'와 '강아지들' 두 단어를 다른 단어로 생각
- FastText는 기존의 Word2Vec 학습 방식은 동일하지만 기준이 되는 문자가 다름
    - Word2Vec은 단어를 기준, FastText 글자를 기준
    - '강아지' -> '강', '아', '지', '강아', '아지', '강아지' (n-gram 방식)
    - subword를 기준으로 문자들을 벡터화
    - Word2Vec 매개변수들을 이용을 하지만 추가적으로 subword 관련한 매개변수들이 사용
        - 최소 길이의 subword -> min_n(기본값은 3)
        - 최대 길이의 subword -> max_n(기본값은 6)
        - min_n = 0, max_n = 0 로 설정을 하면 subword 사용 금지 -> Word2Vec 방식을 이용
        - min_n이 max_n보다 작거나 같게 설정 
        - bucket매개변수는 단어(subword) 사전에 개수를 제한 ( 메모리 과부하 방지 )

In [ ]:
from gensim.models import Word2Vec, FastText

In [ ]:
sentences = [
    ['이커머스', '데이터', '분석', '진행'], 
    ['상품', '리뷰', '감성', '분석', '합니다'], 
    ['형태소', '단위', '임베딩', '가능']
]

In [ ]:
model_wv = Word2Vec(
    sentences = sentences, 
    window = 3, 
    vector_size = 50, 
    min_count = 1, 
    sg  =1, 
    epochs = 10
)

In [ ]:
model_ft = FastText(
    sentences = sentences, 
    window = 3, 
    vector_size = 50, 
    min_count = 1, 
    sg = 1, 
    epochs = 50, 
    min_n = 2, 
    max_n = 4
)

In [ ]:
# 특정 단어의 단위 벡터 출력 
model_wv.wv['이커머스']

In [ ]:
model_ft.wv['이커머스']

In [ ]:
# 단어 간의 유사도 
model_wv.wv.most_similar('데이터', topn=3)

In [ ]:
model_ft.wv.most_similar('데이터', topn=3)

In [ ]:
# 단어 사전에 없는 단어를 이용해서 단위 벡터를 생성한다?
# Word2Vec은 단어 사전에 없는 단어는 단위 벡터화가 불가능
# model_wv.wv['감정']

In [ ]:
# FastText는 새로운 단어에 대한 단위 벡터 생성이 가능
model_ft.wv['감정']

In [ ]:
# 단어에 없는 단어로 유사한 단어를 찾을수 있을까?
model_wv.wv.most_similar('감정', topn=3)

In [ ]:
model_ft.wv.most_similar('감정', topn=3)

In [ ]:
sentences2 = [
    ['고양이', '고양이들', '귀엽다', '동물', '반려동물'], 
    ['강아지', '강아지들', '귀엽다', '동물', '반려동물'], 
    ['달리다', '달리는', '달림', '걷다', '걷는'] ,
    ['예쁘다', '예쁨', '예쁜', '매력적이다'], 
    ['컴퓨터', '컴퓨팅', '컴퓨터들', '기계']
]

In [ ]:
w2v = Word2Vec(
    sentences = sentences2, 
    window = 3, 
    min_count = 1, 
    sg = 1, 
    seed = 42
)
ft = FastText(
    sentences = sentences2, 
    window = 3, 
    min_count = 1, 
    sg = 1, 
    seed = 42, 
    min_n = 2
)

In [ ]:
# 단어 간의 유사도를 확인 
print(
    'Word2Vec 단어 간의 유사도 :', w2v.wv.similarity('강아지', '강아지들')
)
print(
    'FastText 단어 같의 유사도 :', ft.wv.similarity('강아지', '강아지들')
)

In [ ]:
# 비교 대상 단어들
test_text = [
    ['강아지', '강아지들'], 
    ['고양이', '고양이들'], 
    ['달리다', '달리는'], 
    ['예쁘다', '예쁜'], 
    ['컴퓨터', '컴퓨팅'], 
    ['달리다', '걷다']
]

for word1, word2 in test_text:
    w2v_sim = w2v.wv.similarity(word1, word2)
    ft_sim = ft.wv.similarity(word1, word2)
    print(f"{word1}과 {word2} 간의 유사도 : Word2Vec({w2v_sim}) / FastText({ft_sim})")

In [ ]:
# 상품명을 기준으로 특정 상품을 검색 시 연관 된 상품의 목록을 확인 
products = {
    'P001' : '무선 이어폰 블루투스 노이즈캔슬링 충전케이스', 
    'P002' : '유선 이어폰 하이파이 금도금 플러그', 
    'P003' : '게이밍 마우스 RGB 경량 디자인', 
    'P004' : '무선 마우스 초경량 블루투스 듀얼모드', 
    'P005' : '헤드폰 노이즈캔슬링 유선'
}

In [ ]:
# dict형태 데이터에서 FastText에서 필요한 데이터는 values
data = products.values()
data

In [ ]:
from konlpy.tag import Komoran

In [ ]:
komoran = Komoran()

tokens_komoran = []
tokens_split = []
for name in data:
    tokens_komoran.append( komoran.morphs(name) )
    tokens_split.append( name.split() )

In [ ]:
print(tokens_komoran)
print(tokens_split)

In [ ]:
ft2 = FastText(
    sentences = tokens_split, 
    window = 3, 
    min_count = 1, 
    sg = 1, 
    min_n = 2, 
    max_n = 4, 
    seed = 42
)

In [ ]:
# 상품 명들을 벡터화 
# 단어들의 단위 벡터를 구하고 -> 평균을 낸다 -> L2 정규화를 이용하여 거리는 1로 변경 
import numpy as np

In [ ]:
def sent_vector(token, type = None):
    # token : 토큰화된 문장 데이터
    # type : None인 경우는 평규값, 'l2'로 들어오면 L2 정규화
    vectors = []
    for word in token:
        # if word in w2v.wv.index_to_key:
        vectors.append( ft2.wv[word] )
    
    # vectors가 존재하지 않는 경우 
    if not vectors:
        return np.zeros(ft2.vector_size)

    # 단위 벡터들의 평균값을 구한다. 
    v = np.mean(vectors, axis=0)

    # type이 만약에 'l2'라면
    if type == 'l2':
        # 벡터의 거리로 나눠준다. 
        # 벡터의 거리는? ->  numpy의 거리를 구하는 함수 linalg()
        v =  v / ( np.linalg.norm(v) + 1e-12 )

    return v

In [ ]:
# token_split을 이용하여 벡터화 
items_vector = []

for token in tokens_split:
    items_vector.append(
        sent_vector(token)
    )

In [ ]:
items_vector_l2 = []

for token in tokens_split:
    items_vector_l2.append(
        sent_vector(token, 'l2')
    )

In [ ]:
# 문장들의 평균 벡터 -> 특정 상품과 비슷한 상품의 이름을 코사인 유사도를 이용하여 관련 상품 목록을 확인 
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
idx = list(products.keys()).index('P003')
# list 안에 index()함수는? -> 특정 값의 위치를 출력
idx

In [ ]:
items_vector[2]

In [39]:
cosine_similarity([items_vector[idx]], items_vector)

array([[0.01113469, 0.02338488, 1.        , 0.15756822, 0.1412824 ]],
      dtype=float32)

In [ ]:
cosine_similarity( [ft2.wv['마우스']], items_vector )

In [46]:
# 코사인 유사도를 이용해서 가장 근접한 상품의 이름을 출력하는 함수 
def recommand_by_text(product_id):
    # dict.keys() 특정 아이디의 위치를 확인 
    idx = list(products.keys()).index(product_id)

    # 해당 아이디 위치의 단위 벡터 데이터와 다른 상품들의 단위 벡터 데이터를 코사인 유사도를 이용하여 계산
    sims = cosine_similarity( [ items_vector[idx] ], items_vector ).ravel()
    # 내림차순 정렬 (오름차순 정렬을 역순)
    order = sims.argsort()[::-1]
    print(order)
    # 정렬의 순서를 인덱스로 표시
    # 추천 상품명을 담기 위한 빈 리스트 생성
    result = []
    for i in order:
        # 코사인 유사도가 1인 경우 ? -> i가 위치값 -> list(products.keys())[i]이 값이 product_id와 같은 경우 
        # if list(products.keys())[i] != product_id:
        if sims[i] != 1.0:
            result.append(
                [
                    list(products.values())[i], round(sims[i], 4)
                ]
            )
    return result

In [48]:
recommand_by_text('P003')

[2 3 4 1 0]


[['무선 마우스 초경량 블루투스 듀얼모드', np.float32(0.1576)],
 ['헤드폰 노이즈캔슬링 유선', np.float32(0.1413)],
 ['유선 이어폰 하이파이 금도금 플러그', np.float32(0.0234)],
 ['무선 이어폰 블루투스 노이즈캔슬링 충전케이스', np.float32(0.0111)]]

#### 문제 
- 특정한 검색어를 입력 했을때 유사한 상품 목록 3개를 추천하는 함수 를 생성 
- items_vector_l2 l2 정규화가 된 문장 벡터를 이용하여 코사인 유사도를 구하고 상위의 3개의 추천 상품명을 출력
- '노이즈캔슬링' 입력값을 넣엇을때 추천 상품 3개의 결과를 확인 

In [58]:
def recommand_by_search(text):
    # 검색어를 이용하여 등록된 상품 목록에서 유사한 상품을 추천
    # text를 이용해서 단위 벡터를 생성 l2 정규화가 된 벡터데이터와 코사인 유사도를 활용
    vector = ft2.wv[text] 
    v = vector / (np.linalg.norm(vector) + 1e-12 )

    sims = cosine_similarity( [v], items_vector_l2 ).ravel()

    order = sims.argsort()[::-1]
    print(order[:3])
    result = []
    for i in order[:3]:
        result.append(
            [
                list( products.values() )[i], round(sims[i], 4)
            ]
        )
    return result

In [59]:
recommand_by_search('노이즈캔슬링')

[4 0 3]


[['헤드폰 노이즈캔슬링 유선', np.float32(0.4451)],
 ['무선 이어폰 블루투스 노이즈캔슬링 충전케이스', np.float32(0.3391)],
 ['무선 마우스 초경량 블루투스 듀얼모드', np.float32(0.0278)]]